In [ ]:
# ===========================================================================
# ERROR ANALYSIS - interactive driver (Phase 5)
#
# All analysis/plotting logic lives in src/error_analysis.py; this notebook
# only calls it and stores results, matching notebooks/01-03's convention.
#
# Phase 1 tells us HOW OFTEN the model is wrong. This notebook asks WHAT it
# gets wrong and WHY: which speakers and severity groups carry the errors,
# what the most confident failures look like acoustically, and whether the
# misclassified utterances differ measurably (Phase 4's Praat features) from
# the ones the model gets right.
#
# PREREQUISITES
#   1. A trained run, i.e. outputs/predictions/<RUN_NAME>/*.csv exist.
#      Those CSVs must carry a `filename` column - predictions written before
#      src/training/reporting.py started recording utterance identity cannot
#      be joined back to audio, and load_run_predictions will say so.
#   2. outputs/praat_features.csv (Stage 1 of notebooks/03_praat_analysis.ipynb)
#      for the acoustic correlation in Stages 3-4.
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv, print_table
from src.error_analysis import (attach_metadata, compare_error_vs_correct,
                                error_summary, load_run_predictions,
                                most_confident_errors, plot_embedding_map,
                                plot_error_feature_distributions,
                                plot_error_gallery)
from src.training.data import load_manifest

config.ensure_directories()

# Which trained run to analyse. Change this to compare models - the whole
# notebook is keyed off it.
RUN_NAME = "detection_fusion"
TASK = "detection"

df_m6 = load_manifest()
praat_features = (pd.read_csv(config.PRAAT_FEATURES_PATH)
                  if config.PRAAT_FEATURES_PATH.exists() else None)

print_header(f"Error Analysis - {RUN_NAME}")
print_kv("Manifest", f"{len(df_m6)} utterances")
print_kv("Praat features", "loaded" if praat_features is not None
         else "MISSING - run notebooks/03 Stage 1 for Stages 3-4 below")

In [ ]:
# STAGE 1 - Load this run's per-fold predictions and join on the manifest
# (Filepath, WordCode, Severity) plus every Phase 4 Praat feature, keyed by
# filename. `preds` is the single table every stage below reads.
preds = load_run_predictions(RUN_NAME)
preds = attach_metadata(preds, df_m6, praat_features)

print_kv("Rows", len(preds))
print_kv("Columns", len(preds.columns))
preds.head()

In [ ]:
# STAGE 2 - Where do the errors live?
#
# Overall error rate, then broken down by severity group, speaker, true class,
# and word - each sorted worst-first. The severity breakdown is the one that
# usually carries the story: if errors concentrate in 'Very Low' severity, the
# model is failing exactly where dysarthria is subtlest, which is both the
# expected result and the clinically important one to state plainly.
summary = error_summary(preds)

print_header("Overall")
print_table(summary["overall"])

print_header("Error rate by severity group")
print_table(summary["by_severity"].reset_index())

print_header("Error rate by speaker (worst 10)")
print_table(summary["by_speaker"].head(10).reset_index())

print_header("Confusions")
print_table(summary["confusions"])

for name, table in summary.items():
    table.to_csv(config.METRICS_DIR / f"errors_{RUN_NAME}_{name}.csv")
print_kv("Saved", f"{len(summary)} breakdown tables to {config.METRICS_DIR}")

In [ ]:
# STAGE 3 - What do the worst failures actually look like?
#
# The n misclassifications the model was MOST CONFIDENT about. A wrong call at
# p=0.51 is a coin flip and tells us nothing; a wrong call at p=0.99 means the
# model has confidently learned something wrong, and that is worth looking at.
#
# Each gets a 4-panel diagnostic: waveform, spectrogram, MFCC heatmap, and
# Praat F0 contour. Figures land in outputs/figures/errors/<RUN_NAME>/.
worst = most_confident_errors(preds, n=8)
print_header("Most confident misclassifications")
print_table(worst[["filename", "speaker_id", "y_true_label",
                   "y_pred_label", "confidence"]])

gallery = plot_error_gallery(preds, RUN_NAME, n=8, show=True)

In [ ]:
# STAGE 4 - Do the errors share an acoustic signature?
#
# The Phase 5 question, answered directly: for every Praat feature, compare the
# misclassified utterances against the correctly-classified ones (Mann-Whitney U,
# Bonferroni-corrected), ranked by Cliff's delta.
#
# Read the effect size, not the p-value: on ~21k utterances almost anything is
# 'significant', so p alone would be misleading. |delta| >= 0.33 is where a
# difference becomes worth writing about.
comparison = compare_error_vs_correct(preds)
comparison.to_csv(config.METRICS_DIR / f"errors_{RUN_NAME}_feature_comparison.csv",
                  index=False)

print_header("Errors vs correct predictions, by acoustic feature")
print_table(comparison[["feature", "mean_correct", "mean_error", "cliffs_delta",
                        "magnitude", "p_adj", "significant"]].head(12))

figure_path = plot_error_feature_distributions(preds, comparison, RUN_NAME,
                                               top_k=6, show=True)
print_kv("Figure", figure_path)
comparison

In [ ]:
# STAGE 5 - Are the errors scattered, or clustered?
#
# t-SNE of the model's own learned test embeddings, coloured by true class with
# the misclassifications marked. Scattered errors mean genuinely ambiguous
# utterances at the decision boundary. Errors clustered into a coherent region
# mean the model has mislabelled a whole pocket of the space - a much more
# actionable finding, and one an accuracy number can never show.
embedding_figure = plot_embedding_map(RUN_NAME, preds, task=TASK, show=True)

print_header("Phase 5 - Error Analysis complete")
print_kv("Embedding map", embedding_figure)
print_kv("Diagnostics", f"{len(gallery)} figure(s) in {config.ERROR_FIGURE_DIR / RUN_NAME}")
print_kv("Tables", f"outputs/metrics/errors_{RUN_NAME}_*.csv")